In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Load test data and best model
df = pd.read_csv('../data/processed/featured_data.csv')
best_model = joblib.load('../data/models/best_model.pkl')
scaler = joblib.load('../data/models/scaler.pkl')

# Prepare test data
feature_cols = [col for col in df.columns if col not in ['timestamp', 'target_temp_24h', 
                                                           'weather_description', 'country',
                                                           'latitude', 'longitude']]
split_idx = int(len(df) * 0.8)
X_test = df[feature_cols][split_idx:]
y_test = df['target_temp_24h'][split_idx:]
X_test_scaled = scaler.transform(X_test)

# Make predictions
y_pred = best_model.predict(X_test_scaled)

# Create results dataframe
results_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred,
    'error': y_test.values - y_pred,
    'abs_error': np.abs(y_test.values - y_pred)
})

print("=== PREDICTION RESULTS ===")
print(f"Mean Absolute Error: {results_df['abs_error'].mean():.3f}°C")
print(f"Median Absolute Error: {results_df['abs_error'].median():.3f}°C")
print(f"Max Error: {results_df['abs_error'].max():.3f}°C")
print(f"90th Percentile Error: {results_df['abs_error'].quantile(0.9):.3f}°C")

# 1. Actual vs Predicted scatter plot
plt.figure(figsize=(10, 8))
plt.scatter(results_df['actual'], results_df['predicted'], alpha=0.6)
plt.plot([results_df['actual'].min(), results_df['actual'].max()], 
         [results_df['actual'].min(), results_df['actual'].max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Temperature (°C)')
plt.ylabel('Predicted Temperature (°C)')
plt.title('Actual vs Predicted Temperature')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/actual_vs_predicted.png', dpi=300)
plt.show()

# 2. Residuals plot
plt.figure(figsize=(10, 6))
plt.scatter(results_df['predicted'], results_df['error'], alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Temperature (°C)')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Residual Plot')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/residuals.png', dpi=300)
plt.show()

# 3. Error distribution
plt.figure(figsize=(10, 6))
plt.hist(results_df['error'], bins=30, edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='r', linestyle='--', lw=2, label='Zero Error')
plt.xlabel('Prediction Error (°C)')
plt.ylabel('Frequency')
plt.title('Distribution of Prediction Errors')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/error_distribution.png', dpi=300)
plt.show()

# 4. Time series of predictions
plt.figure(figsize=(14, 6))
sample_size = min(100, len(results_df))  # Plot last 100 predictions
plt.plot(range(sample_size), results_df['actual'].tail(sample_size), 
         marker='o', label='Actual', alpha=0.7)
plt.plot(range(sample_size), results_df['predicted'].tail(sample_size), 
         marker='s', label='Predicted', alpha=0.7)
plt.xlabel('Time Steps')
plt.ylabel('Temperature (°C)')
plt.title('Actual vs Predicted Temperature Over Time (Last 100 Predictions)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/predictions_over_time.png', dpi=300)
plt.show()

print("\n Evaluation complete! Plots saved")

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

# Save feature names
feature_names = feature_cols
joblib.dump(feature_names, '../data/models/feature_names.pkl')

# Save model metadata
import json
metadata = {
    'model_type': type(best_model).__name__,
    'features': feature_names,
    'train_size': split_idx,
    'test_size': len(X_test),
    'test_mae': float(results_df['abs_error'].mean()),
    'test_r2': float(r2_score(y_test, y_pred)),
    'date_trained': pd.Timestamp.now().isoformat()
}

with open('../data/models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Model artifacts saved")